# QB Performance Deep Dive — 2025 Season

A detailed analysis of quarterback efficiency, accuracy, and situational performance
across the 2025 NFL regular season. We look at passing accuracy by depth, performance
under pressure, turnover tendencies, clutch play, and completion over expected (CPOE).

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
# Load data
df = pd.read_parquet('../data/processed/pass_plays_qualified.parquet')
features = pd.read_parquet('../data/processed/qb_features.parquet')

# Throws only (no sacks/scrambles) for accuracy analysis
throws = df[(df['sack'] == 0) & (df['qb_scramble'] == 0)].copy()

print(f'Total pass plays: {len(df):,}')
print(f'Total throws: {len(throws):,}')
print(f'Qualifying QBs: {df.passer_player_name.nunique()}')

Total pass plays: 16,335
Total throws: 15,243
Qualifying QBs: 38


## 1. Overall Passing Efficiency Rankings

EPA per play is the gold standard for measuring QB efficiency — it captures
the expected point value added on every single play.

In [3]:
# EPA per play rankings
efficiency = df.groupby('passer_player_name').agg(
    attempts=('play_id', 'count'),
    epa_per_play=('epa', 'mean'),
    total_epa=('epa', 'sum'),
    comp_pct=('complete_pass', lambda x: x[df.loc[x.index, 'sack'] == 0].mean()),
).sort_values('epa_per_play', ascending=True)

# Highlight colors — make one QB stand out
colors = ['#e74c3c' if 'Hurts' in name else '#3498db' for name in efficiency.index]

fig = go.Figure(go.Bar(
    x=efficiency['epa_per_play'],
    y=efficiency.index,
    orientation='h',
    marker_color=colors,
    text=[f"{v:.3f}" for v in efficiency['epa_per_play']],
    textposition='outside',
))
fig.update_layout(
    title='EPA per Play — All Qualifying QBs (2025 Regular Season)',
    xaxis_title='EPA per Play',
    width=900, height=800,
    plot_bgcolor='white',
    xaxis=dict(zeroline=True, zerolinecolor='black', zerolinewidth=1),
)
fig.show()

# Print rank
ranked = efficiency['epa_per_play'].rank(ascending=False)
for name in efficiency.index:
    if 'Hurts' in name:
        print(f"\n{name}: EPA/play rank: {int(ranked[name])} of {len(ranked)}")


J.Hurts: EPA/play rank: 17 of 38


## 2. Completion Accuracy by Depth

Breaking down completion percentage by throw depth reveals accuracy on short,
medium, and deep passes separately — not all completion percentages are created equal.

In [4]:
# Completion % by depth zone
throws_clean = throws[throws['air_yards'].notna()].copy()
throws_clean['depth_zone'] = pd.cut(
    throws_clean['air_yards'],
    bins=[-np.inf, 10, 20, np.inf],
    labels=['Short (<10)', 'Medium (10-20)', 'Deep (20+)']
)

depth_comp = throws_clean.groupby(['passer_player_name', 'depth_zone'])['complete_pass'].agg(
    ['mean', 'count']
).reset_index()
depth_comp.columns = ['QB', 'Depth', 'Comp%', 'Attempts']

# Filter to QBs with enough attempts at each level
depth_comp = depth_comp[depth_comp['Attempts'] >= 15]

for zone in ['Short (<10)', 'Medium (10-20)', 'Deep (20+)']:
    zone_data = depth_comp[depth_comp['Depth'] == zone].sort_values('Comp%', ascending=True)
    
    colors = ['#e74c3c' if 'Hurts' in name else '#3498db' for name in zone_data['QB']]
    
    fig = go.Figure(go.Bar(
        x=zone_data['Comp%'],
        y=zone_data['QB'],
        orientation='h',
        marker_color=colors,
        text=[f"{v:.1%}" for v in zone_data['Comp%']],
        textposition='outside',
    ))
    fig.update_layout(
        title=f'Completion % — {zone} Passes',
        xaxis_title='Completion %',
        width=900, height=700,
        plot_bgcolor='white',
        xaxis=dict(tickformat='.0%'),
    )
    fig.show()
    
    # Print rank
    for _, row in zone_data.iterrows():
        if 'Hurts' in row['QB']:
            rank = zone_data['Comp%'].rank(ascending=False)
            hurts_rank = int(rank[zone_data['QB'] == row['QB']].values[0])
            print(f"{row['QB']} — {zone}: {row['Comp%']:.1%} (rank {hurts_rank} of {len(zone_data)})")

J.Hurts — Short (<10): 75.0% (rank 9 of 38)


J.Hurts — Medium (10-20): 46.8% (rank 29 of 38)


J.Hurts — Deep (20+): 37.7% (rank 13 of 36)


## 3. CPOE — Completion % Over Expected

CPOE adjusts for throw difficulty. A QB with a low CPOE completes fewer passes
than the model expects given the air yards, down & distance, and situation of each throw.
This is arguably the purest measure of passing accuracy.

In [5]:
# Use the pre-computed CPOE from nflverse (built into the play-by-play data)
cpoe_data = throws[throws['cpoe'].notna()].groupby('passer_player_name').agg(
    avg_cpoe=('cpoe', 'mean'),
    throws=('cpoe', 'count'),
).sort_values('avg_cpoe', ascending=True)

colors = ['#e74c3c' if 'Hurts' in name else 
          '#2ecc71' if val >= 0 else '#95a5a6' 
          for name, val in zip(cpoe_data.index, cpoe_data['avg_cpoe'])]
# Override color for Hurts
colors = ['#e74c3c' if 'Hurts' in name else c for name, c in zip(cpoe_data.index, colors)]

fig = go.Figure(go.Bar(
    x=cpoe_data['avg_cpoe'],
    y=cpoe_data.index,
    orientation='h',
    marker_color=colors,
    text=[f"{v:+.1f}%" for v in cpoe_data['avg_cpoe']],
    textposition='outside',
))
fig.update_layout(
    title='CPOE — Completion % Over Expected (2025 Season)',
    xaxis_title='CPOE (positive = better than expected)',
    width=900, height=800,
    plot_bgcolor='white',
    xaxis=dict(zeroline=True, zerolinecolor='black', zerolinewidth=1),
)
fig.show()

ranked = cpoe_data['avg_cpoe'].rank(ascending=False)
for name in cpoe_data.index:
    if 'Hurts' in name:
        print(f"\n{name}: CPOE rank: {int(ranked[name])} of {len(ranked)}")
        print(f"CPOE: {cpoe_data.loc[name, 'avg_cpoe']:+.2f}%")


J.Hurts: CPOE rank: 9 of 38
CPOE: +3.18%


## 4. Turnover Analysis

Interceptions and turnover-worthy plays — how often does each QB put the ball in danger?

In [6]:
# Interception rate
turnovers = throws.groupby('passer_player_name').agg(
    attempts=('play_id', 'count'),
    interceptions=('interception', 'sum'),
).copy()
turnovers['int_rate'] = turnovers['interceptions'] / turnovers['attempts']
turnovers = turnovers.sort_values('int_rate', ascending=False)

colors = ['#e74c3c' if 'Hurts' in name else '#3498db' for name in turnovers.index]

fig = go.Figure(go.Bar(
    x=turnovers.index,
    y=turnovers['int_rate'],
    marker_color=colors,
    text=[f"{v:.1%}" for v in turnovers['int_rate']],
    textposition='outside',
))
fig.update_layout(
    title='Interception Rate by QB (2025 Season)',
    yaxis_title='Interception Rate',
    xaxis_tickangle=-45,
    width=1000, height=600,
    plot_bgcolor='white',
    yaxis=dict(tickformat='.1%'),
)
fig.show()

ranked = turnovers['int_rate'].rank(ascending=False)
for name in turnovers.index:
    if 'Hurts' in name:
        print(f"\n{name}: INT rate rank: {int(ranked[name])} of {len(ranked)} (1 = most INTs)")
        print(f"INTs: {int(turnovers.loc[name, 'interceptions'])} on {int(turnovers.loc[name, 'attempts'])} attempts ({turnovers.loc[name, 'int_rate']:.1%})")


J.Hurts: INT rate rank: 33 of 38 (1 = most INTs)
INTs: 6 on 452 attempts (1.3%)


## 5. Performance Under Pressure

How well does each QB perform when the pass rush gets home? Pressure resilience
separates elite passers from the rest.

In [7]:
# EPA under pressure
pressure_plays = throws[throws['was_pressure'].notna()].copy()

pressured = pressure_plays[pressure_plays['was_pressure'] == 1]
clean = pressure_plays[pressure_plays['was_pressure'] == 0]

pressure_stats = pd.DataFrame({
    'epa_pressured': pressured.groupby('passer_player_name')['epa'].mean(),
    'epa_clean': clean.groupby('passer_player_name')['epa'].mean(),
    'comp_pressured': pressured.groupby('passer_player_name')['complete_pass'].mean(),
    'comp_clean': clean.groupby('passer_player_name')['complete_pass'].mean(),
    'n_pressured': pressured.groupby('passer_player_name')['play_id'].count(),
})

# Filter to QBs with enough pressured throws
pressure_stats = pressure_stats[pressure_stats['n_pressured'] >= 30].copy()
pressure_stats['epa_drop'] = pressure_stats['epa_pressured'] - pressure_stats['epa_clean']

# Plot EPA under pressure (worst to best)
pressure_stats = pressure_stats.sort_values('epa_pressured', ascending=True)

colors = ['#e74c3c' if 'Hurts' in name else '#3498db' for name in pressure_stats.index]

fig = go.Figure(go.Bar(
    x=pressure_stats['epa_pressured'],
    y=pressure_stats.index,
    orientation='h',
    marker_color=colors,
    text=[f"{v:.3f}" for v in pressure_stats['epa_pressured']],
    textposition='outside',
))
fig.update_layout(
    title='EPA per Play Under Pressure (2025 Season)',
    xaxis_title='EPA/Play When Pressured',
    width=900, height=700,
    plot_bgcolor='white',
    xaxis=dict(zeroline=True, zerolinecolor='black', zerolinewidth=1),
)
fig.show()

ranked = pressure_stats['epa_pressured'].rank(ascending=False)
for name in pressure_stats.index:
    if 'Hurts' in name:
        print(f"\n{name} under pressure:")
        print(f"  EPA/play: {pressure_stats.loc[name, 'epa_pressured']:.3f} (rank {int(ranked[name])} of {len(ranked)})")
        print(f"  Comp%: {pressure_stats.loc[name, 'comp_pressured']:.1%}")
        print(f"  EPA drop from clean pocket: {pressure_stats.loc[name, 'epa_drop']:.3f}")


J.Hurts under pressure:
  EPA/play: -0.368 (rank 34 of 38)
  Comp%: 43.1%
  EPA drop from clean pocket: -0.700


## 6. Sack Rate

Sack rate reflects both offensive line play and a QB's ability to get the ball out
or escape pressure. High sack rates kill drives.

In [8]:
# Sack rate (sacks / dropbacks)
dropbacks = df[df['qb_dropback'] == 1]
sack_stats = dropbacks.groupby('passer_player_name').agg(
    dropbacks=('play_id', 'count'),
    sacks=('sack', 'sum'),
)
sack_stats['sack_rate'] = sack_stats['sacks'] / sack_stats['dropbacks']
sack_stats = sack_stats.sort_values('sack_rate', ascending=False)

colors = ['#e74c3c' if 'Hurts' in name else '#3498db' for name in sack_stats.index]

fig = go.Figure(go.Bar(
    x=sack_stats.index,
    y=sack_stats['sack_rate'],
    marker_color=colors,
    text=[f"{v:.1%}" for v in sack_stats['sack_rate']],
    textposition='outside',
))
fig.update_layout(
    title='Sack Rate by QB (2025 Season)',
    yaxis_title='Sack Rate (Sacks / Dropbacks)',
    xaxis_tickangle=-45,
    width=1000, height=600,
    plot_bgcolor='white',
    yaxis=dict(tickformat='.1%'),
)
fig.show()

ranked = sack_stats['sack_rate'].rank(ascending=False)
for name in sack_stats.index:
    if 'Hurts' in name:
        print(f"\n{name}: Sack rate rank: {int(ranked[name])} of {len(ranked)} (1 = most sacked)")
        print(f"Sacks: {int(sack_stats.loc[name, 'sacks'])} on {int(sack_stats.loc[name, 'dropbacks'])} dropbacks ({sack_stats.loc[name, 'sack_rate']:.1%})")


J.Hurts: Sack rate rank: 19 of 38 (1 = most sacked)
Sacks: 32 on 484 dropbacks (6.6%)


## 7. Clutch Performance — 4th Quarter, Close Games

How does each QB perform when the game is on the line? We define clutch situations
as 4th quarter plays with the score within 8 points.

In [9]:
clutch = df[(df['qtr'] == 4) & (df['score_differential'].abs() <= 8)].copy()

clutch_stats = clutch.groupby('passer_player_name').agg(
    clutch_plays=('play_id', 'count'),
    clutch_epa=('epa', 'mean'),
    clutch_comp=('complete_pass', lambda x: x[clutch.loc[x.index, 'sack'] == 0].mean()),
    clutch_int=('interception', 'sum'),
    clutch_td=('touchdown', 'sum'),
)

# Filter to QBs with enough clutch plays
clutch_stats = clutch_stats[clutch_stats['clutch_plays'] >= 20].copy()
clutch_stats['clutch_td_int_ratio'] = clutch_stats['clutch_td'] / clutch_stats['clutch_int'].clip(lower=1)
clutch_stats = clutch_stats.sort_values('clutch_epa', ascending=True)

colors = ['#e74c3c' if 'Hurts' in name else '#3498db' for name in clutch_stats.index]

fig = go.Figure(go.Bar(
    x=clutch_stats['clutch_epa'],
    y=clutch_stats.index,
    orientation='h',
    marker_color=colors,
    text=[f"{v:.3f}" for v in clutch_stats['clutch_epa']],
    textposition='outside',
))
fig.update_layout(
    title='Clutch EPA — 4th Quarter, Score Within 8 Points',
    xaxis_title='EPA/Play in Clutch Situations',
    width=900, height=700,
    plot_bgcolor='white',
    xaxis=dict(zeroline=True, zerolinecolor='black', zerolinewidth=1),
)
fig.show()

ranked = clutch_stats['clutch_epa'].rank(ascending=False)
for name in clutch_stats.index:
    if 'Hurts' in name:
        print(f"\n{name} in clutch situations:")
        print(f"  EPA/play: {clutch_stats.loc[name, 'clutch_epa']:.3f} (rank {int(ranked[name])} of {len(ranked)})")
        print(f"  TDs: {int(clutch_stats.loc[name, 'clutch_td'])}, INTs: {int(clutch_stats.loc[name, 'clutch_int'])}")


J.Hurts in clutch situations:
  EPA/play: -0.019 (rank 21 of 36)
  TDs: 3, INTs: 1


## 8. Negative Play Rate

What percentage of a QB's plays result in negative EPA? This captures
how often a QB actively hurts his team on a given play.

In [10]:
neg_play = df.groupby('passer_player_name').agg(
    total_plays=('epa', 'count'),
    negative_plays=('epa', lambda x: (x < 0).sum()),
)
neg_play['neg_play_rate'] = neg_play['negative_plays'] / neg_play['total_plays']
neg_play = neg_play.sort_values('neg_play_rate', ascending=False)

colors = ['#e74c3c' if 'Hurts' in name else '#3498db' for name in neg_play.index]

fig = go.Figure(go.Bar(
    x=neg_play.index,
    y=neg_play['neg_play_rate'],
    marker_color=colors,
    text=[f"{v:.1%}" for v in neg_play['neg_play_rate']],
    textposition='outside',
))
fig.update_layout(
    title='Negative EPA Play Rate (2025 Season)',
    yaxis_title='% of Plays with Negative EPA',
    xaxis_tickangle=-45,
    width=1000, height=600,
    plot_bgcolor='white',
    yaxis=dict(tickformat='.0%'),
)
fig.show()

ranked = neg_play['neg_play_rate'].rank(ascending=False)
for name in neg_play.index:
    if 'Hurts' in name:
        print(f"\n{name}: Negative play rate rank: {int(ranked[name])} of {len(ranked)} (1 = worst)")
        print(f"{neg_play.loc[name, 'neg_play_rate']:.1%} of plays produce negative EPA")


J.Hurts: Negative play rate rank: 12 of 38 (1 = worst)
56.6% of plays produce negative EPA


## 9. Weekly EPA Trend

How consistent is each QB across the season? Plotting weekly EPA reveals
cold streaks, hot streaks, and overall trajectory.

In [11]:
# Weekly EPA for selected QBs
weekly = df.groupby(['passer_player_name', 'week'])['epa'].mean().reset_index()

# Pick comparison QBs — top performers + Hurts
top_qbs_by_epa = df.groupby('passer_player_name')['epa'].mean().nlargest(5).index.tolist()
highlight_qbs = list(set(top_qbs_by_epa + ['J.Hurts']))

weekly_filtered = weekly[weekly['passer_player_name'].isin(highlight_qbs)]

fig = px.line(
    weekly_filtered, x='week', y='epa',
    color='passer_player_name',
    title='Weekly EPA per Play — Top QBs vs J.Hurts',
    labels={'epa': 'EPA/Play', 'week': 'Week', 'passer_player_name': 'QB'},
    markers=True,
)
fig.update_layout(
    width=1000, height=550,
    plot_bgcolor='white',
    hovermode='x unified',
)
# Make Hurts line thicker and dashed
for trace in fig.data:
    if 'Hurts' in trace.name:
        trace.line.width = 4
        trace.line.dash = 'dash'
        trace.line.color = '#e74c3c'

fig.add_hline(y=0, line_dash='dot', line_color='gray', opacity=0.5)
fig.show()

## 10. Summary Report Card

In [12]:
# Build a summary table ranking a specific QB across all metrics
def rank_qb(qb_name, df, throws):
    """Generate a full ranking report card for a QB."""
    results = {}
    
    # EPA/play
    epa_ranks = df.groupby('passer_player_name')['epa'].mean()
    n = len(epa_ranks)
    results['EPA/Play'] = (epa_ranks[qb_name], int(epa_ranks.rank(ascending=False)[qb_name]), n)
    
    # Completion %
    comp = throws.groupby('passer_player_name')['complete_pass'].mean()
    results['Completion %'] = (comp[qb_name], int(comp.rank(ascending=False)[qb_name]), len(comp))
    
    # CPOE
    cpoe = throws[throws['cpoe'].notna()].groupby('passer_player_name')['cpoe'].mean()
    if qb_name in cpoe.index:
        results['CPOE'] = (cpoe[qb_name], int(cpoe.rank(ascending=False)[qb_name]), len(cpoe))
    
    # INT rate
    int_rate = throws.groupby('passer_player_name')['interception'].mean()
    results['INT Rate'] = (int_rate[qb_name], int(int_rate.rank(ascending=True)[qb_name]), len(int_rate))
    
    # Sack rate
    db = df[df['qb_dropback'] == 1]
    sack_r = db.groupby('passer_player_name')['sack'].mean()
    results['Sack Rate'] = (sack_r[qb_name], int(sack_r.rank(ascending=True)[qb_name]), len(sack_r))
    
    # Neg play rate
    neg_r = df.groupby('passer_player_name')['epa'].apply(lambda x: (x < 0).mean())
    results['Neg Play Rate'] = (neg_r[qb_name], int(neg_r.rank(ascending=True)[qb_name]), len(neg_r))
    
    # Clutch EPA
    clutch_df = df[(df['qtr'] == 4) & (df['score_differential'].abs() <= 8)]
    clutch_epa = clutch_df.groupby('passer_player_name')['epa'].mean()
    if qb_name in clutch_epa.index:
        results['Clutch EPA'] = (clutch_epa[qb_name], int(clutch_epa.rank(ascending=False)[qb_name]), len(clutch_epa))
    
    # Format output
    print(f"\n{'='*55}")
    print(f"  {qb_name} — 2025 Season Report Card")
    print(f"{'='*55}")
    print(f"{'Metric':<20} {'Value':>10} {'Rank':>12}")
    print(f"{'-'*55}")
    for metric, (val, rank, total) in results.items():
        if 'Rate' in metric or 'Completion' in metric:
            val_str = f"{val:.1%}"
        elif 'CPOE' in metric:
            val_str = f"{val:+.2f}%"
        else:
            val_str = f"{val:.3f}"
        rank_str = f"{rank} of {total}"
        print(f"{metric:<20} {val_str:>10} {rank_str:>12}")

rank_qb('J.Hurts', df, throws)


  J.Hurts — 2025 Season Report Card
Metric                    Value         Rank
-------------------------------------------------------
EPA/Play                  0.057     17 of 38
Completion %              65.0%     19 of 38
CPOE                     +3.18%      9 of 38
INT Rate                   1.3%      6 of 38
Sack Rate                  6.6%     20 of 38
Neg Play Rate             56.6%     27 of 38
Clutch EPA               -0.019     22 of 38
